# Baseline-модель предпочтений (логистическая регрессия) для синтетического датасета

Что делаем:
1) Загружаем `profiles_23band.csv` и `ab_pairs.csv` из `datasets/synth_quick/`
2) Строим признаки **по 6 перцептивным параметрам**: `bass, tilt, presence, air, lowmid, sparkle`
3) Превращаем A/B сравнения в задачу **pairwise classification**:
   - признак = `x_A - x_B`
   - таргет = `y_a_wins` (1 если A лучше B)
4) Обучаем Logistic Regression (baseline)
5) Считаем первые метрики (accuracy, logloss, AUC)
6) Подбираем гиперпараметры (по `C`) и показываем улучшение
7) Сохраняем модель на диск

Выход:
- `models/baseline_pairwise_logreg.joblib`


In [1]:
import os
from pathlib import Path
import numpy as np
import pandas as pd

DATA_DIR = Path("datasets/synth_quick_V1")
PROFILES_PATH = DATA_DIR / "profiles_23band.csv"
PAIRS_PATH = DATA_DIR / "ab_pairs.csv"

assert PROFILES_PATH.exists(), f"Не найден файл: {PROFILES_PATH}"
assert PAIRS_PATH.exists(), f"Не найден файл: {PAIRS_PATH}"

profiles = pd.read_csv(PROFILES_PATH)
pairs = pd.read_csv(PAIRS_PATH)

print("profiles:", profiles.shape)
print("pairs:", pairs.shape)
profiles.head()


profiles: (300, 32)
pairs: (2500, 5)


,profile_id,archetype,preamp_db,bass,tilt,presence,air,lowmid,sparkle,g_20,...,g_1486,g_1875,g_2368,g_3390,g_4365,g_6934,g_8569,g_12000,g_14000,g_16000
0,0,warm,-6.221703,1.005000,-0.121742,0.137620,-0.376347,0.501192,-0.079049,-1.484424,...,-6.205469,-6.144570,-6.078300,-6.212423,-6.535455,-7.253329,-7.535341,-7.853617,-7.932036,-7.970643
1,1,vshape,-4.893288,0.688115,-0.164630,-0.723607,0.503700,0.423229,0.096959,-1.235452,...,-5.499645,-6.243776,-7.050842,-7.232092,-6.355611,-4.592447,-4.144014,-3.798668,-3.752925,-3.753665
2,2,bright,-5.653092,-0.889004,-0.017210,0.056964,0.905178,0.048929,0.877672,-9.206988,...,-5.498612,-5.367244,-5.186489,-4.849936,-4.503446,-3.061524,-2.070409,-1.007760,-1.000000,-1.215161
3,3,neutral,-1.031580,-0.161823,0.300292,-0.066957,-0.446490,-0.396651,-0.321808,-3.225021,...,-1.000000,-1.009218,-1.051379,-1.091985,-1.114932,-1.475701,-1.787990,-2.100981,-2.064385,-1.948286
4,4,warm,-4.381094,0.749849,0.041670,-0.249538,-0.344511,0.618957,-0.408271,-1.521881,...,-4.599150,-4.876564,-5.195480,-5.395789,-5.289919,-5.459421,-5.812024,-6.223923,-6.205260,-6.089370


## 1) Подготовка признаков (6 параметров)

Мы учимся **не по 23 точкам кривой**, а по компактному вектору из 6 параметров — это и есть “компактный EQ” для модели.


In [2]:
FEATURES = ["bass", "tilt", "presence", "air", "lowmid", "sparkle"]

missing = [c for c in FEATURES if c not in profiles.columns]
assert not missing, f"В profiles нет колонок: {missing}"

X_profiles = profiles.set_index("profile_id")[FEATURES].astype(float)
print("X_profiles:", X_profiles.shape)
X_profiles.head()


X_profiles: (300, 6)


,bass,tilt,presence,air,lowmid,sparkle
profile_id,,,,,,
0,1.005000,-0.121742,0.137620,-0.376347,0.501192,-0.079049
1,0.688115,-0.164630,-0.723607,0.503700,0.423229,0.096959
2,-0.889004,-0.017210,0.056964,0.905178,0.048929,0.877672
3,-0.161823,0.300292,-0.066957,-0.446490,-0.396651,-0.321808
4,0.749849,0.041670,-0.249538,-0.344511,0.618957,-0.408271


## 2) Перевод A/B в обучение классификатора

Для каждой пары (A,B) строим разность признаков: `x = x_A - x_B`.
Тогда логистическая регрессия учит вектор вкуса `w` такой, что:

`P(A лучше B) = sigmoid( w · (x_A - x_B) )`

Это классический baseline для pairwise preference learning (Bradley–Terry в форме логрегрессии).


In [3]:
def build_pairwise_dataset(pairs_df: pd.DataFrame, X_profiles: pd.DataFrame):
    a = pairs_df["a_id"].to_numpy()
    b = pairs_df["b_id"].to_numpy()
    y = pairs_df["y_a_wins"].to_numpy().astype(int)

    Xa = X_profiles.loc[a].to_numpy()
    Xb = X_profiles.loc[b].to_numpy()
    X = Xa - Xb
    return X, y

# split
train = pairs[pairs["split"] == "train"].reset_index(drop=True)
val   = pairs[pairs["split"] == "val"].reset_index(drop=True)
test  = pairs[pairs["split"] == "test"].reset_index(drop=True)

X_train, y_train = build_pairwise_dataset(train, X_profiles)
X_val, y_val     = build_pairwise_dataset(val, X_profiles)
X_test, y_test   = build_pairwise_dataset(test, X_profiles)

X_train.shape, X_val.shape, X_test.shape


((1600, 6), (400, 6), (500, 6))

## 3) Baseline: Logistic Regression

Метрики:
- accuracy
- log loss
- ROC-AUC (необязательно, но полезно)

Важно: для логлосса и AUC используем вероятности `predict_proba`.


In [7]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, log_loss, roc_auc_score

def eval_model(clf, X, y, name=""):
    p = clf.predict_proba(X)[:, 1]
    pred = (p >= 0.5).astype(int)
    return {
        "split": name,
        "accuracy": float(accuracy_score(y, pred)),
        "logloss": float(log_loss(y, p)),
        "auc": float(roc_auc_score(y, p)),
    }

baseline = LogisticRegression(
    C=1.0,
    l1_ratio=0.0,        # L2-регуляризация
    solver="liblinear",
    max_iter=200,
    random_state=42,
)

baseline.fit(X_train, y_train)

metrics = [
    eval_model(baseline, X_train, y_train, "train"),
    eval_model(baseline, X_val, y_val, "val"),
    eval_model(baseline, X_test, y_test, "test"),
]
pd.DataFrame(metrics)


,split,accuracy,logloss,auc
0,train,0.66375,0.598307,0.736026
1,val,0.66250,0.622559,0.711260
2,test,0.66600,0.589029,0.749595


## 4) Подбор гиперпараметров (простая сетка по C)

Делаем максимально просто и быстро:
- перебираем несколько значений `C`
- выбираем по минимальному logloss на validation
- после выбора обучаем финальную модель на train+val


In [9]:
from sklearn.base import clone

C_grid = [0.01, 0.03, 0.1, 0.3, 1.0, 3.0, 10.0, 30.0, 100.0]

rows = []
best = None
best_val = 1e9

for C in C_grid:
    clf = LogisticRegression(
        C=1.0,
        l1_ratio=0.0,
        solver="liblinear",
        max_iter=200,
        random_state=42,
    )
    clf.fit(X_train, y_train)
    m_val = eval_model(clf, X_val, y_val, "val")
    rows.append({"C": C, **m_val})
    if m_val["logloss"] < best_val:
        best_val = m_val["logloss"]
        best = clf

grid_df = pd.DataFrame(rows).sort_values("logloss")
grid_df


,C,split,accuracy,logloss,auc
0,0.01,val,0.6625,0.622559,0.71126
1,0.03,val,0.6625,0.622559,0.71126
2,0.10,val,0.6625,0.622559,0.71126
3,0.30,val,0.6625,0.622559,0.71126
4,1.00,val,0.6625,0.622559,0.71126
5,3.00,val,0.6625,0.622559,0.71126
6,10.00,val,0.6625,0.622559,0.71126
7,30.00,val,0.6625,0.622559,0.71126
8,100.00,val,0.6625,0.622559,0.71126


In [11]:
best_C = float(grid_df.iloc[0]["C"])
print("best C:", best_C)

# финальная модель на train+val
X_trainval = np.vstack([X_train, X_val])
y_trainval = np.hstack([y_train, y_val])

final_model = LogisticRegression(
    C=1.0,
    l1_ratio=0.0,
    solver="liblinear",
    max_iter=200,
    random_state=42,
)
final_model.fit(X_trainval, y_trainval)

final_metrics = [
    eval_model(final_model, X_train, y_train, "train"),
    eval_model(final_model, X_val, y_val, "val"),
    eval_model(final_model, X_test, y_test, "test"),
]
pd.DataFrame(final_metrics)


best C: 0.01


,split,accuracy,logloss,auc
0,train,0.661875,0.598622,0.735657
1,val,0.670000,0.619992,0.713235
2,test,0.668000,0.590157,0.749935


In [12]:
base_test = eval_model(baseline, X_test, y_test, "baseline_test")
tuned_test = eval_model(final_model, X_test, y_test, "tuned_test")

pd.DataFrame([base_test, tuned_test])


,split,accuracy,logloss,auc
0,baseline_test,0.666,0.589029,0.749595
1,tuned_test,0.668,0.590157,0.749935


In [13]:
w = final_model.coef_[0]
weights = pd.DataFrame({"feature": FEATURES, "weight": w}).sort_values("weight", ascending=False)
weights


,feature,weight
0,bass,0.942140
3,air,0.678324
5,sparkle,0.467318
2,presence,0.314750
1,tilt,-0.132269
4,lowmid,-0.267106


In [14]:
import joblib

MODEL_DIR = Path("models")
MODEL_DIR.mkdir(parents=True, exist_ok=True)

bundle = {
    "model": final_model,
    "features": FEATURES,
    "best_C": best_C,
}

out_path = MODEL_DIR / "baseline_pairwise_logreg.joblib"
joblib.dump(bundle, out_path)

print("saved:", out_path.resolve())


saved: C:\Users\makcc\PycharmProjects\EarLoop\research\experiment\models\baseline_pairwise_logreg.joblib
